In [1]:
BASE_URL = "https://simkatmawa.kemdiktisaintek.go.id"
CONFIG_PATH = r"c:\Users\ACER\Downloads\sertifikasi_runner\config.ini"

import configparser
import requests

config = configparser.ConfigParser()
config.read(CONFIG_PATH, encoding='utf-8')
email = config.get('credentials', 'email')
password = config.get('credentials', 'password')

print(f"[*] Melakukan login API untuk: {email} ...")
resp = requests.post(f"{BASE_URL}/api/login", json={"email": email, "password": password})
if resp.status_code == 200:
    data = resp.json()
    tokentot = data.get('token')
    print("  [+] Login Berhasil! Token didapatkan.")
else:
    print("  [-] Login Gagal:", resp.text)
    tokentot = ""


In [2]:
import csv
import json
import requests
import argparse
import sys

In [3]:
def baca_csv(filepath: str) -> list[dict]:
    """Baca file CSV dengan delimiter titik koma dan kembalikan list baris."""
    rows = []
    with open(filepath, newline="", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f, delimiter=";")
        for row in reader:
            cleaned = {}
            for k, v in row.items():
                key = k.strip() if k else k
                val = v.strip().strip('"') if v else ""
                cleaned[key] = val
            rows.append(cleaned)
    print(f"[CSV] Membaca {len(rows)} baris dari '{filepath}'.")
    return rows

In [ ]:
def baris_ke_payload(row: dict) -> dict:
    """Konversi satu baris CSV menjadi payload JSON untuk API."""
    level_raw = row.get("level", "INT").strip().upper()

    return {
        "level":                level_raw,
        "nama":                 row.get("nama_sertifikasi", ""),
        "penyelenggara":        row.get("nama_penyelenggara", ""),
        "url_peserta":          row.get("url_rekognisi", ""),
        "url_sertifikat":       row.get("url_sertif", ""),
        "tgl_sertifikat":       row.get("tgl_sertif", ""),
        "url_foto_upp":         "-",   # dikosongkan sesuai permintaan
        "url_dokumen_undangan": "-",
        "keterangan":           "-",
        "mahasiswa": [
            {
                "nim":  row.get("nim", ""),
                "nama": row.get("nama_mahasiswa", ""),
            }
        ],
        "dosen": [
            {
                "nuptk":          row.get("nidn", ""),
                "nama":           row.get("nama_dosen", ""),
                "url_surat_tugas": row.get("surat_tugas", ""),
            }
        ],
    }

In [5]:
def upload_sertifikasi(token: str, payload: dict, idx: int) -> str:
    """Kirim satu payload ke endpoint POST /api/sertifikasi."""
    url     = f"{BASE_URL}/api/sertifikasi"
    headers = {
        "Content-Type":  "application/json",
        "Authorization": f"Bearer {token}",
    }

    nama_mhs = payload["mahasiswa"][0]["nama"] if payload["mahasiswa"] else "-"
    print(f"\n[UPLOAD #{idx}] '{payload['nama']}' — {nama_mhs}")

    try:
        resp = requests.post(url, json=payload, headers=headers, timeout=30)
        if resp.status_code in (200, 201):
            print(f"  ✅ Berhasil! Status: {resp.status_code}")
            try:
                print(f"  Response: {resp.json()}")
                #get data.id dari response
                return resp.json().get("data", {}).get("id", "N/A")

            except Exception:
                print(f"  Response: {resp.text[:200]}")
            return "N/A"
        else:
            print(f"  ❌ Gagal! Status: {resp.status_code}")
            print(f"  Response: {resp.text[:500]}")
            return "N/A"
    except requests.exceptions.RequestException as e:
        print(f"  ❌ Request error: {e}")
        return False

In [6]:
csv = baca_csv("edu.csv")

[CSV] Membaca 524 baris dari 'edu.csv'.


In [7]:
for idx, row in enumerate(csv, start=1):
    payload = baris_ke_payload(row)
    print(f"\n[PAYLOAD #{idx}] {json.dumps(payload, indent=2)}")


[PAYLOAD #1] {
  "level": "NAS",
  "nama": "Sertifikasi Kompetensi Bahasa Inggris (Edu First)",
  "penyelenggara": "PT. Edu First Solusindo",
  "url_peserta": "https://www.edufirst.id/",
  "url_sertifikat": "https://drive.google.com/file/d/1G04XV7lsGiJfwejTNqI_fumHb4WCWfUL/view?usp=sharing",
  "tgl_sertifikat": "2025-06-25",
  "url_foto_upp": "-",
  "url_dokumen_undangan": "-",
  "keterangan": "-",
  "mahasiswa": [
    {
      "nim": "11000124140746",
      "nama": "Christian Simanjutak"
    }
  ],
  "dosen": [
    {
      "nuptk": "0009078401",
      "nama": "Dr. Aditya Yuli Sulistyawan, S.H., M.H",
      "url_surat_tugas": "https://drive.google.com/file/d/1qQDXQeQlPFaysaXT8RpVDa-Sse04Oi0z/view?usp=drive_link"
    }
  ]
}

[PAYLOAD #2] {
  "level": "NAS",
  "nama": "Sertifikasi Kompetensi Bahasa Inggris (Edu First)",
  "penyelenggara": "PT. Edu First Solusindo",
  "url_peserta": "https://www.edufirst.id/",
  "url_sertifikat": "https://drive.google.com/file/d/1bcZ99_VJs9NW1bKD7I7wmJAZ

In [8]:
#tampilkan per kolom berapa banyak data yang kosong
print("\n[CEK KOLOM KOSONG]")

kolom_kosong = {}
for idx, row in enumerate(csv, start=1):
    for k, v in row.items():
        if not v.strip():
            kolom_kosong[k] = kolom_kosong.get(k, 0) + 1
for kolom, count in kolom_kosong.items():
    print(f"  Kolom '{kolom}': {count} data kosong")


    


[CEK KOLOM KOSONG]
  Kolom 'id (kosongi)': 524 data kosong
  Kolom 'url_foto': 524 data kosong
  Kolom '': 524 data kosong


In [9]:
#cari yang nimnya kosong atau not found
print("\n[CEK NIM KOSONG]")
for idx, row in enumerate(csv, start=1):
    nim = row.get("nim", "").strip()
    if not nim:
        print(f"  Baris #{idx} - Nama: '{row.get('nama_mahasiswa', '-')}' - NIM: '{nim}'")


[CEK NIM KOSONG]


In [10]:
#tampilkan unique value dari nama sertifikat

print("\n[UNIQUE NAMA SERTIFIKAT]")
unique_sertifikat = set()
for row in csv:
    nama_sertifikat = row.get("nama sertifikasi", "").strip()
    if nama_sertifikat:
        unique_sertifikat.add(nama_sertifikat)

print(f"  Total unique nama sertifikat: {len(unique_sertifikat)}")
for idx, nama in enumerate(sorted(unique_sertifikat), start=1):
    print(f"  {idx}. {nama} (Total: {sum(1 for r in csv if r.get('nama sertifikasi', '').strip() == nama)})")



[UNIQUE NAMA SERTIFIKAT]
  Total unique nama sertifikat: 1
  1. Sertifikasi Kompetensi Bahasa Inggris (Edu First) (Total: 524)


In [11]:
#TES UPLOAD 2 DATA SAJA
print("\n[UPLOAD TEST 2 DATA]")
for idx, row in enumerate(csv[:2], start=1):
    payload = baris_ke_payload(row)
    upload_sertifikasi(tokentot, payload, idx)


[UPLOAD TEST 2 DATA]

[UPLOAD #1] 'Sertifikasi Kompetensi Bahasa Inggris (Edu First)' — Christian Simanjutak
  ✅ Berhasil! Status: 200
  Response: {'status': True, 'message': 'Sertifikasi berhasil disimpan', 'data': {'level': 'NAS', 'nama': 'Sertifikasi Kompetensi Bahasa Inggris (Edu First)', 'penyelenggara': 'PT. Edu First Solusindo', 'url_peserta': 'https://www.edufirst.id/', 'url_sertifikat': 'https://drive.google.com/file/d/1G04XV7lsGiJfwejTNqI_fumHb4WCWfUL/view?usp=sharing', 'tgl_sertifikat': '2025-06-25', 'url_foto_upp': '-', 'url_dokumen_undangan': '-', 'kode_pt': '001008', 'source': 'API', 'tahun': '2025', 'updated_at': '2026-03-07T02:11:34.000000Z', 'created_at': '2026-03-07T02:11:34.000000Z', 'id': 132566}}

[UPLOAD #2] 'Sertifikasi Kompetensi Bahasa Inggris (Edu First)' — Tegar Abdillah
  ✅ Berhasil! Status: 200
  Response: {'status': True, 'message': 'Sertifikasi berhasil disimpan', 'data': {'level': 'NAS', 'nama': 'Sertifikasi Kompetensi Bahasa Inggris (Edu First)', 'peny

In [12]:
#UPLOAD DATA KE API, PASTIKAN DATA SUDAH BENAR, TIDAK ADA NIM KOSONG, DLL

from datetime import datetime

log_file = "upload_log_{}.csv".format(datetime.now().strftime("%Y-%m-%d_%H-%M-%S"))

with open(log_file, "w") as f:
    f.write("idx,nama_mhs,sertifikasi_id\n")

for idx, row in enumerate(csv, start=1):
    payload = baris_ke_payload(row)
    print(f"\n[PAYLOAD #{idx}] {json.dumps(payload, indent=2)}")
    sertifikat_id = upload_sertifikasi(tokentot, payload, idx)
    # simpan hasil upload ke file log
    with open(log_file, "a") as f:
        f.write(f"{idx},{payload['mahasiswa'][0]['nama'] if payload['mahasiswa'] else '-'},{sertifikat_id}\n")



[PAYLOAD #1] {
  "level": "NAS",
  "nama": "Sertifikasi Kompetensi Bahasa Inggris (Edu First)",
  "penyelenggara": "PT. Edu First Solusindo",
  "url_peserta": "https://www.edufirst.id/",
  "url_sertifikat": "https://drive.google.com/file/d/1G04XV7lsGiJfwejTNqI_fumHb4WCWfUL/view?usp=sharing",
  "tgl_sertifikat": "2025-06-25",
  "url_foto_upp": "-",
  "url_dokumen_undangan": "-",
  "keterangan": "-",
  "mahasiswa": [
    {
      "nim": "11000124140746",
      "nama": "Christian Simanjutak"
    }
  ],
  "dosen": [
    {
      "nuptk": "0009078401",
      "nama": "Dr. Aditya Yuli Sulistyawan, S.H., M.H",
      "url_surat_tugas": "https://drive.google.com/file/d/1qQDXQeQlPFaysaXT8RpVDa-Sse04Oi0z/view?usp=drive_link"
    }
  ]
}

[UPLOAD #1] 'Sertifikasi Kompetensi Bahasa Inggris (Edu First)' — Christian Simanjutak
  ✅ Berhasil! Status: 200
  Response: {'status': True, 'message': 'Sertifikasi berhasil disimpan', 'data': {'level': 'NAS', 'nama': 'Sertifikasi Kompetensi Bahasa Inggris (Edu Fi